In [35]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
import numpy as np
import time
from collections import defaultdict
from scipy.optimize import linprog

In [36]:
def mpformulate_theta_bounds(flex_sol, num_theta:int, theta_bounds:list, num_design:int=0, design_bounds:list=None, psi_idx:int=0, theta_m:int=0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty((len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx,:num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    print(f'num_cr:{num_cr}')
    print(f'num_theta:{num_theta}')
    print(f'num_design:{num_design}')
    print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")
    
    c = np.hstack([np.array([-1, 1]).reshape(1, -1), np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1,1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')
    
    row1_block = np.hstack([block for i in range(theta_m, num_theta) for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta) for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1), np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    print(f'A: {A}')
    print(f'A.shape: {A.shape}')
    
    x_lb = np.array([val for i in range(theta_m, len(theta_bounds)) for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds)) for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1,1)), -x_lb.reshape(-1,1), x_ub.reshape(-1,1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')
    
    if F0.size==0 and theta_m==0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([]) 
    
    F = np.vstack([F0, F0, np.zeros((1,num_design)), np.zeros((4*(num_theta-theta_m), num_design))]) if num_design>0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1,len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1,len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    
    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')
    
    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')
    
    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list) 
                                                                        else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list) 
                                                                        else [])).reshape(-1, 1)
    
    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')
    
    return A, b, c, H, A_t, b_t, F

In [37]:
def get_theta_bounds(flex_sol, numt, tbounds, numd:int=0, dbounds:list=None):
    
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)
    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(flex_sol=flex_sol, num_theta=numt ,num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        if F.size != 0:
            prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
            prob.process_constraints()
            solution = solve_mpqp(problem=prob, algorithm=mpqp_algorithm.geometric)
            prob_dict[f't{i}'] = prob
            theta_bound_dict[f't{i}'] = solution
        else:
            linsol = linprog(c=c, A_ub=A, b_ub=b)
            prob_dict[f't{i}'] = linsol
            theta_bound_dict[f't{i}'] = [linsol.x[1], linsol.x[0]]
            # if linsol.success:
                # print("Optimal value:", linsol.fun)
                # print("Optimal x:", linsol.x)
        print(f'Finished solving for theta{i+1}')
    probs = [p for key, p in prob_dict.items()]
    sols = [sol for key, sol in theta_bound_dict.items()]
    
    return probs, sols
    

In [56]:
y1_val = 1
y2_val = 1
y3_val = 1

t_bounds = [(8, 16), (3, 11)]
d_bounds = [(3, 7), (7, 9), (7, 9)]
nt = len(t_bounds)
nd = len(d_bounds)

j1 = 0.92
j2 = 0.85
j3 = 0.75

m = MPModeler()

u = m.add_var(name='u')
F1 = m.add_var(name="F1")
F2 = m.add_var(name="F2")
F3 = m.add_var(name="F3")
F4 = m.add_var(name="F4")
F5 = m.add_var(name="F5")
F6 = m.add_var(name="F6")
F7 = m.add_var(name="F7")

S = m.add_param(name='S')
D = m.add_param(name='D')

d1 = m.add_param(name='d1')
d2 = m.add_param(name='d2')
d3 = m.add_param(name='d3')


m.add_constr(F4 - j1 * F2 == 0)
m.add_constr(F1 - F2 - F3 == 0)
m.add_constr(F5 - j2 * F4 == 0)
m.add_constr(F6 - j3 * F3 == 0)
m.add_constr(F7 - F5 - F6 == 0)
m.add_constr(F1 - S <= u)
m.add_constr(D - F7 <= u)
m.add_constr(F2 - d1 * y1_val <= u)
m.add_constr(F4 - d2 * y2_val <= u)
m.add_constr(F3 - d3 * y3_val <= u)

for v in [F1, F2, F3, F4, F5, F6, F7]:
    m.add_constr(v >= 0)

m.add_constr(t_bounds[0][0] + 1e-6 <= S)
m.add_constr(S <= t_bounds[0][1])
m.add_constr(t_bounds[1][0] + 1e-6 <= D)
m.add_constr(D <= t_bounds[1][1])

m.add_constr(d_bounds[0][0] <= d1)
m.add_constr(d1 <= d_bounds[0][1])
m.add_constr(d_bounds[1][0] <= d2)
m.add_constr(d2 <= d_bounds[1][1])
m.add_constr(d_bounds[2][0] <= d3)
m.add_constr(d3 <= d_bounds[2][1])

m.set_objective(u)

prob = m.formulate_problem()
prob.process_constraints()

solution_flexibility = solve_mpqp(
    problem=prob, algorithm=mpqp_algorithm.geometric)

# start_time = time.time()
# prob_list, sol_list = get_theta_bounds(
#     flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds)
# end_time = time.time()
# print(f'Elapsed time for solving mp problems: {end_time-start_time}')
# 
# 
# def joint_pdf(theta: list):
#     Sval, Dval = theta
#     eps = 1e-12
#     # x = max(Sval - 8.0, eps)
#     return (1/(1.2 * np.pi * (Sval - 8.0))) * np.exp(
#         -1.39*(np.log(Sval - 8.0))**2 - 0.5*(Dval - 7.0)**2
#     )

Using a found active set [0, 1, 2, 3, 4, 6, 7, 8]


In [58]:
len(solution_flexibility.critical_regions)

2

In [54]:
m1 =MPModeler()

Smax = m1.add_var(name='Smax')
Smin = m1.add_var(name='Smin')
Da = m1.add_var(name='Da')
Db = m1.add_var(name='Db')

d1 = m1.add_param(name='d1')
d2 = m1.add_param(name='d2')
d3 = m1.add_param(name='d3')

m1.add_constr(-0.421*Smax + 0.561*Da - 0.018*d1 <= 0)
m1.add_constr(0.395*Da - 0.309*d1 - 0.296*d3 <=0)
m1.add_constr(-0.421*Smin + 0.561*Db - 0.018*d1 <= 0)
m1.add_constr(0.395*Db - 0.309*d1 - 0.296*d3 <=0)
m1.add_constr(Smin <= Smax)

m1.add_constr(t_bounds[0][0] + 1e-6 <= Smax)
m1.add_constr(Smax <= t_bounds[0][1])
m1.add_constr(t_bounds[1][0] + 1e-6 <= Da)
m1.add_constr(Da <= t_bounds[1][1])

m1.add_constr(t_bounds[0][0] + 1e-6 <= Smin)
m1.add_constr(Smin <= t_bounds[0][1])
m1.add_constr(t_bounds[1][0] + 1e-6 <= Db)
m1.add_constr(Db <= t_bounds[1][1])

m1.add_constr(d_bounds[0][0] <= d1)
m1.add_constr(d1 <= d_bounds[0][1])
m1.add_constr(d_bounds[1][0] <= d2)
m1.add_constr(d2 <= d_bounds[1][1])
m1.add_constr(d_bounds[2][0] <= d3)
m1.add_constr(d3 <= d_bounds[2][1])

m1.set_objective(Smin-Smax)

prob_Sbounds = m1.formulate_problem()
prob_Sbounds.process_constraints()

solution_Sbounds = solve_mpqp(
    problem=prob_Sbounds, algorithm=mpqp_algorithm.geometric)

Using a found active set [6, 7, 9, 11]


In [55]:
for cr in solution_Sbounds.critical_regions:
    print(f'Smax : {cr.A[0], cr.b[0]}, Smin : {cr.A[1], cr.b[1]}')

Smax : (array([0., 0., 0.]), array([16.])), Smin : (array([0., 0., 0.]), array([8.000001]))


In [41]:
prob_Sbounds.A.shape

(13, 4)

In [42]:
prob_Sbounds.b.shape

(13, 1)

In [43]:
prob_Sbounds.F.shape

(13, 3)

In [44]:
len(solution_flexibility.critical_regions)

2

In [45]:
A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(flex_sol=solution_flexibility, num_theta=2 ,num_design=3, theta_bounds=t_bounds, design_bounds=d_bounds, theta_m=0)

num_cr:2
num_theta:2
num_design:3
A0: [[-0.42087542  0.56116723]
 [ 0.          0.39494471]]
A: [[-0.42087542  0.          0.56116723  0.        ]
 [ 0.          0.          0.39494471  0.        ]
 [ 0.         -0.42087542  0.          0.56116723]
 [ 0.          0.          0.          0.39494471]
 [-1.          1.          0.          0.        ]
 [-1.         -0.         -0.         -0.        ]
 [-0.         -1.         -0.         -0.        ]
 [-0.         -0.         -1.         -0.        ]
 [-0.         -0.         -0.         -1.        ]
 [ 1.          0.          0.          0.        ]
 [ 0.          1.          0.          0.        ]
 [ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]]
A.shape: (13, 4)


In [46]:
prob_test = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
prob_test.process_constraints()

In [47]:
prob_test.A.shape

(13, 4)

In [48]:
prob_Sbounds.A

array([[-0.60003054,  0.        ,  0.79956563,  0.        ],
       [ 0.        ,  0.        ,  0.67829578,  0.        ],
       [ 0.        , -0.60003054,  0.        ,  0.79956563],
       [ 0.        ,  0.        ,  0.        ,  0.67829578],
       [-0.70710678,  0.70710678,  0.        ,  0.        ],
       [-1.        ,  0.        ,  0.        ,  0.        ],
       [ 1.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -1.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        , -1.        ,  0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -1.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [49]:
prob_test.A

array([[-0.59980349,  0.        ,  0.79973798,  0.        ],
       [ 0.        ,  0.        ,  0.67821569,  0.        ],
       [ 0.        , -0.59980349,  0.        ,  0.79973798],
       [ 0.        ,  0.        ,  0.        ,  0.67821569],
       [-0.70710678,  0.70710678,  0.        ,  0.        ],
       [-1.        , -0.        , -0.        , -0.        ],
       [-0.        , -1.        , -0.        , -0.        ],
       [-0.        , -0.        , -1.        , -0.        ],
       [-0.        , -0.        , -0.        , -1.        ],
       [ 1.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [52]:
solution_test = solve_mpqp(
    problem=prob_test, algorithm=mpqp_algorithm.geometric)

Using a found active set [6, 7, 8, 9]


In [53]:
solution_test.critical_regions

[Critical region with active set [6, 7, 8, 9]
 The Omega Constraint indices are [0, 1, 2, 3, 4, 5]
 The Lagrange multipliers Constraint indices are []
 The Regular Constraint indices are [[], []]
   x(θ) = Aθ + b 
  λ(θ) = Cθ + d 
   Eθ <= f
  A = [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]] 
  b = [[16.]
  [ 8.]
  [ 3.]
  [ 3.]] 
  C = [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]] 
  d = [[1.]
  [0.]
  [0.]
  [1.]] 
  E = [[-1. -0. -0.]
  [-0. -1. -0.]
  [-0. -0. -1.]
  [ 1.  0.  0.]
  [ 0.  1.  0.]
  [ 0.  0.  1.]] 
  f = [[-3.]
  [-7.]
  [-7.]
  [ 7.]
  [ 9.]
  [ 9.]]]